# DNS TTL and Cache Hit Rate

Companion notebook for the [DNS Resolution End to End](https://codemia.io/courses/networking_for_backend_engineers/net_dns_resolution_end_to_end) lesson on Codemia.

**What you will build**: a cache simulator for a stream of lookups, the hit rate a given time to
live produces, and the propagation delay the same time to live costs you during a failover. The two
pull in opposite directions, and the plot at the end is the trade written down.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1234)

## Concept recap

A resolver that answers from cache answers in microseconds. A resolver that has to walk the
hierarchy answers in tens or hundreds of milliseconds, and that walk sits in front of the first
connection a client makes. The time to live on a record decides how often that walk happens.

A long time to live means a high hit rate and fast lookups, and it also means that when you change
the record, clients keep using the old answer for up to that long. A short one means the opposite.
Neither is a safety setting: both are a choice about which cost you would rather pay.

## One cache, one name

In [ ]:
def simulate_cache(rate_per_second, ttl_seconds, duration_seconds=3600, rng=rng):
    """Poisson arrivals against a single cached record. Returns (hits, misses, arrival count)."""
    n = rng.poisson(rate_per_second * duration_seconds)
    arrivals = np.sort(rng.uniform(0, duration_seconds, size=n))

    expiry = -np.inf
    hits = 0
    for t in arrivals:
        if t < expiry:
            hits += 1
        else:
            expiry = t + ttl_seconds      # a miss refreshes the record
    return hits, len(arrivals) - hits, len(arrivals)

for rate in [0.05, 0.5, 5, 50]:
    hits, misses, total = simulate_cache(rate, ttl_seconds=60)
    print(f'{rate:>6.2f} lookups/s, TTL 60s -> {hits/total:>6.1%} hits '
          f'({misses:,} misses in an hour, one every {3600/max(misses,1):>6.1f}s)')

The miss count barely moves. That is the first thing to internalise about a cache in front of a
single record: once the arrival rate is well above one per time-to-live interval, the number of
authoritative lookups is set by the time to live alone, not by traffic. Ten times the traffic
produces the same number of misses and a better hit rate.

In [ ]:
def analytic_hit_rate(rate_per_second, ttl_seconds):
    """With Poisson arrivals, one miss starts each TTL interval and the rest are hits."""
    expected_per_interval = rate_per_second * ttl_seconds
    return expected_per_interval / (expected_per_interval + 1)

print(f"{'rate/s':>8}{'TTL':>7}{'simulated':>12}{'analytic':>11}")
for rate in [0.05, 0.5, 5, 50]:
    for ttl in [30, 300]:
        hits, _, total = simulate_cache(rate, ttl)
        print(f'{rate:>8.2f}{ttl:>7}{hits/total:>12.1%}{analytic_hit_rate(rate, ttl):>11.1%}')

The closed form is close enough to use on a whiteboard. It also tells you the shape: the hit rate
depends on the product of the rate and the time to live, so halving the time to live and doubling
the traffic leaves it unchanged.

In [ ]:
ttls = np.logspace(0, 4, 80)

plt.figure(figsize=(9, 5))
for rate, label in [(0.1, '6 lookups/min'), (1, '1/s'), (10, '10/s'), (100, '100/s')]:
    plt.plot(ttls, [analytic_hit_rate(rate, t) for t in ttls], label=label)

for ttl, name in [(30, '30s'), (300, '5min'), (3600, '1h'), (86400, '1 day')]:
    plt.axvline(ttl, color='grey', linestyle=':', linewidth=0.8)
    plt.text(ttl * 1.05, 0.04, name, color='grey', fontsize=8, rotation=90)

plt.xscale('log')
plt.xlabel('time to live (seconds)')
plt.ylabel('cache hit rate')
plt.title('Hit rate depends on rate x TTL, nothing else')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## What the lookups cost

In [ ]:
MISS_MS = 45        # one walk of the hierarchy, cold

def mean_lookup_ms(hit_rate, hit_ms=0.2, miss_ms=MISS_MS):
    return hit_rate * hit_ms + (1 - hit_rate) * miss_ms

def first_percentile_affected(hit_rate):
    """A miss rate of m means the slowest m of requests pay it, so it sits at p(100 - 100m)."""
    miss_rate = 1 - hit_rate
    for label, threshold in [('p90', 0.10), ('p99', 0.01), ('p99.9', 0.001), ('p99.99', 0.0001)]:
        if miss_rate >= threshold:
            return label
    return 'beyond p99.99'

print(f"{'TTL':>8}{'hit rate':>11}{'mean lookup':>14}{'miss rate':>12}{'visible from':>15}")
for ttl in [5, 30, 60, 300, 3600]:
    hr = analytic_hit_rate(2, ttl)      # a modest 2 lookups per second
    print(f'{ttl:>8}{hr:>11.2%}{mean_lookup_ms(hr):>12.2f} ms{1 - hr:>12.3%}'
          f'{first_percentile_affected(hr):>15}')
print()
print(f'every miss costs about {MISS_MS} ms, whatever the mean says')

The mean is reassuring and the mean is not the problem. A miss is a fixed, large cost paid by one
unlucky request, so it lands in the tail rather than in the average. The last column is the one to
read: at a 5-second time to live nearly a tenth of lookups miss, so the 45ms walk is inside the
90th percentile and every dashboard will show it. Lengthen the record to five minutes and the same
45ms is still paid, by so few requests that it hides below the 99.9th percentile.

This is why a service whose own latency budget is 50ms can be destroyed by a short time to live
even though its average resolution cost looks like a rounding error.

## The other side: how long a change takes to land

In [ ]:
def propagation(ttl_seconds, resolvers=10_000, horizon=None, rng=rng):
    """Resolvers hold a record for a uniformly distributed remaining TTL when the change lands."""
    horizon = horizon or ttl_seconds * 1.2
    remaining = rng.uniform(0, ttl_seconds, size=resolvers)
    times = np.linspace(0, horizon, 200)
    switched = [(remaining <= t).mean() for t in times]
    return times, np.array(switched)

print(f"{'TTL':>8}{'half switched':>16}{'99% switched':>15}{'all switched':>15}")
for ttl in [30, 60, 300, 3600, 86400]:
    times, switched = propagation(ttl)
    half = times[np.argmax(switched >= 0.5)]
    p99 = times[np.argmax(switched >= 0.99)]
    print(f'{ttl:>8}{half:>14,.0f} s{p99:>13,.0f} s{ttl:>13,.0f} s')

Half of the fleet follows you in half a time-to-live interval, and the last stragglers take the
whole of it. That is the optimistic version, because it assumes every resolver honours the value;
some clamp a short time to live upward, and an application that resolves once at start-up and keeps
the address forever honours nothing at all.

In [ ]:
plt.figure(figsize=(9, 5))
for ttl, label in [(30, 'TTL 30s'), (300, 'TTL 5min'), (3600, 'TTL 1h')]:
    times, switched = propagation(ttl, horizon=4200)
    plt.plot(times / 60, switched, label=label)

plt.axhline(0.99, color='grey', linestyle='--', linewidth=0.8)
plt.text(1, 1.005, '99% of resolvers', color='grey', fontsize=9)
plt.xlabel('minutes after the record changes')
plt.ylabel('fraction using the new answer')
plt.title('A failover is only as fast as the TTL you set yesterday')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Variations: pick a number for a real service

In [ ]:
scenarios = [
    ('public website, stable address',  200.0, 3600),
    ('API behind a load balancer',      500.0,  300),
    ('record used for failover',        500.0,   30),
    ('internal service, low traffic',     0.2,   60),
]

print(f"{'scenario':<34}{'rate/s':>8}{'TTL':>7}{'hit rate':>10}{'misses/hr':>11}{'failover':>11}")
for name, rate, ttl in scenarios:
    hr = analytic_hit_rate(rate, ttl)
    misses = 3600 / ttl
    print(f'{name:<34}{rate:>8.1f}{ttl:>7}{hr:>10.2%}{misses:>11,.0f}{ttl/60:>9,.1f}m')

Read the last two columns together. The failover column is the worst case you have accepted; the
misses column is what you pay for it, and for a busy name it is small in absolute terms. A record
that fronts a failover pair should have a short time to live because the cost of the extra lookups
is a handful per minute, while the cost of a slow failover is an outage. A record that never changes
should have a long one, because there is nothing to gain by re-asking.

## What to try next

- Add a second tier of cache: a stub resolver on the host in front of the recursive resolver. Two
  caches in series multiply their miss rates, and the hit rate the authoritative server sees is the
  product.
- Model negative caching. A name that does not exist is cached too, often with a different and
  longer interval, which is why a newly created record can be invisible for longer than its own
  time to live.
- Replace the Poisson arrivals with a diurnal pattern and look at what the hit rate does overnight,
  when the rate falls below one lookup per interval and the cache stops helping at all.